In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# Global plot style
matplotlib.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 16,
    'axes.labelweight': 'medium',
    'xtick.labelsize': 14,
    'ytick.labelsize': 14
})

In [ ]:
# Paths and output directory
DATA_PATH = "../1_data/processed/zone_sequence_merged.csv"
SAVE_DIR = "plots/eda_climate"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load merged dataset
climate_fire_df = pd.read_csv(DATA_PATH)
climate_fire_df["Date"] = pd.to_datetime(climate_fire_df["Date"])
climate_fire_df["Month"] = climate_fire_df["Date"].dt.month
climate_fire_df["Year"] = climate_fire_df["Date"].dt.year

zones = sorted(climate_fire_df["Zone_ID"].unique())
months = [3, 4, 5, 6, 7, 8, 9, 10]
month_labels = ["Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]
years = list(range(2008, 2019))

df = climate_fire_df[climate_fire_df["Month"].between(3, 10)].copy()

In [ ]:
# Helper to plot multi-panel zone climate variable
def plot_zone_climate(variable, ylabel, color, filename, ylim=None):
    monthly_zone = (df.groupby(["Zone_ID", "Month"])[variable]
                    .mean().unstack(fill_value=0).reindex(columns=months, fill_value=0))
    max_y = monthly_zone.values.max()
    min_y = monthly_zone.values.min()

    fig, axes = plt.subplots(2, 4, figsize=(22, 12), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, z in enumerate(zones):
        vals = monthly_zone.loc[z].values
        axes[i].plot(range(len(months)), vals, color=color, linewidth=1.5, marker="o", markersize=3)
        axes[i].set_title(f"Zone {z}", fontsize=20, pad=10)
        axes[i].set_xticks(range(len(months)))
        axes[i].set_xticklabels(month_labels, fontsize=14)
        axes[i].grid(axis="y", linestyle="--", alpha=0.6)
        if ylim:
            axes[i].set_ylim(*ylim)
        else:
            axes[i].set_ylim(min_y - 0.05*(max_y-min_y), max_y + 0.05*(max_y-min_y))

    fig.text(0.5, 0.01, "Month", ha="center", fontsize=22)
    fig.text(0.01, 0.5, ylabel, va="center", rotation="vertical", fontsize=22)
    plt.subplots_adjust(left=0.06, bottom=0.09, right=0.98, top=0.93, wspace=0.10, hspace=0.25)
    plt.savefig(os.path.join(SAVE_DIR, filename), dpi=600, bbox_inches="tight")
    plt.close()

In [ ]:
# Helper to plot regional climate variable
def plot_region_climate(variable, ylabel, color, filename, ylim=None):
    monthly_region = (df.groupby("Month")[variable]
                      .mean().reindex(months, fill_value=0))
    max_y = monthly_region.max()
    min_y = monthly_region.min()

    fig, ax = plt.subplots(figsize=(22, 12))
    ax.plot(range(len(months)), monthly_region.values, color=color, linewidth=1.5, marker="o", markersize=3)
    ax.set_xticks(range(len(months)))
    ax.set_xticklabels(month_labels, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    if ylim:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(min_y - 0.05*(max_y-min_y), max_y + 0.05*(max_y-min_y))

    fig.text(0.5, 0.06, "Month", ha="center", fontsize=22)
    fig.text(0.01, 0.5, ylabel, va="center", rotation="vertical", fontsize=22)
    plt.subplots_adjust(left=0.06, bottom=0.14, right=0.98, top=0.93)
    plt.savefig(os.path.join(SAVE_DIR, filename), dpi=600, bbox_inches="tight")
    plt.close()

In [ ]:
# Helper to plot yearly zone climate variable
def plot_zone_yearly_climate(variable, ylabel, color, filename, ylim=None):
    yearly_zone = (climate_fire_df.groupby(["Zone_ID", "Year"])[variable]
                   .mean().unstack(fill_value=0).reindex(columns=years, fill_value=0))
    max_y = yearly_zone.values.max()
    min_y = yearly_zone.values.min()

    fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, z in enumerate(zones):
        vals = yearly_zone.loc[z].values
        axes[i].plot(years, vals, color=color, linewidth=1.5, marker="o", markersize=3)
        axes[i].set_title(f"Zone {z}", fontsize=20, pad=10)
        axes[i].set_xticks(years)
        axes[i].set_xticklabels(years, rotation=45, fontsize=14)
        axes[i].grid(axis="y", linestyle="--", alpha=0.6)
        if ylim:
            axes[i].set_ylim(*ylim)
        else:
            axes[i].set_ylim(min_y - 0.05*(max_y-min_y), max_y + 0.05*(max_y-min_y))

    fig.text(0.5, 0.01, "Year", ha="center", fontsize=18)
    fig.text(0.01, 0.5, ylabel, va="center", rotation="vertical", fontsize=18)
    plt.subplots_adjust(left=0.06, bottom=0.12, right=0.98, top=0.93, wspace=0.15, hspace=0.25)
    plt.savefig(os.path.join(SAVE_DIR, filename), dpi=600, bbox_inches="tight")
    plt.close()

In [ ]:
# Helper to plot yearly regional climate variable
def plot_region_yearly_climate(variable, ylabel, color, filename, ylim=None):
    yearly_region = (climate_fire_df.groupby("Year")[variable]
                     .mean().reindex(range(2008, 2019), fill_value=0))
    max_y = yearly_region.max()
    min_y = yearly_region.min()

    fig, ax = plt.subplots(figsize=(20, 10))
    ax.plot(years, yearly_region.values, color=color, linewidth=1.5, marker="o", markersize=5)
    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=45, fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    if ylim:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(min_y - 0.05*(max_y-min_y), max_y + 0.05*(max_y-min_y))

    fig.text(0.5, 0.06, "Year", ha="center", fontsize=18)
    fig.text(0.01, 0.5, ylabel, va="center", rotation="vertical", fontsize=18)
    plt.subplots_adjust(left=0.06, bottom=0.16, right=0.98, top=0.93)
    plt.savefig(os.path.join(SAVE_DIR, filename), dpi=600, bbox_inches="tight")
    plt.close()

In [ ]:
# Monthly average temperature
plot_zone_climate("Temperature", "Temperature (°C)", "#a51fb4", "monthly_avg_temperature_zones.png")
plot_region_climate("Temperature", "Temperature (°C)", "#a51fb4", "monthly_avg_temperature_region.png")

In [ ]:
# Monthly average humidity
plot_zone_climate("Humidity", "Humidity (%)", "#149fbe", "monthly_avg_humidity_zones.png", ylim=(30, 80))
plot_region_climate("Humidity", "Humidity (%)", "#149fbe", "monthly_avg_humidity_region.png", ylim=(30, 80))

In [ ]:
# Monthly average precipitation
plot_zone_climate("Precipitation", "Precipitation (mm)", "#37bd0e", "monthly_avg_precipitation_zones.png")
plot_region_climate("Precipitation", "Precipitation (mm)", "#37bd0e", "monthly_avg_precipitation_region.png")


In [ ]:
# Monthly average wind speed
plot_zone_climate("Wind", "Wind Speed (km/h)", "#090989", "monthly_avg_wind_zones.png")
plot_region_climate("Wind", "Wind Speed (km/h)", "#090989", "monthly_avg_wind_region.png")

In [ ]:
# Yearly average temperature
plot_zone_yearly_climate("Temperature", "Temperature (°C)", "#a51fb4", "yearly_avg_temperature_zones.png")
plot_region_yearly_climate("Temperature", "Temperature (°C)", "#a51fb4", "yearly_avg_temperature_region.png")

In [ ]:
# Yearly average humidity
plot_zone_yearly_climate("Humidity", "Humidity (%)", "#149fbe", "yearly_avg_humidity_zones.png", ylim=(30, 80))
plot_region_yearly_climate("Humidity", "Humidity (%)", "#149fbe", "yearly_avg_humidity_region.png", ylim=(30, 80))

In [ ]:
# Yearly average precipitation
plot_zone_yearly_climate("Precipitation", "Precipitation (mm/day)", "#37bd0e", "yearly_avg_precipitation_zones.png")
plot_region_yearly_climate("Precipitation", "Precipitation (mm/day)", "#37bd0e", "yearly_avg_precipitation_region.png")

In [ ]:
# Yearly average wind speed
plot_zone_yearly_climate("Wind", "Wind Speed (km/h)", "#090989", "yearly_avg_wind_zones.png")
plot_region_yearly_climate("Wind", "Wind Speed (km/h)", "#090989", "yearly_avg_wind_region.png")